# Fire Weather ML Predictions
This notebook trains the same three models used by the standalone Flask service. It intentionally excludes K-Means. The final dashboard does not depend on Colab: train once, save the model files, then load them from `ml_service.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, classification_report, roc_auc_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATASET = Path('/content/drive/MyDrive/ByteSmart Internship/jena_climate_2009_2016.csv')
MODEL_DIR = Path('/content/drive/MyDrive/ByteSmart Internship/fire_weather_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## Load and prepare the time series
The pressure-rate feature is calculated **before sampling**. This fixes the old notebook bug where shifting 18 sampled rows no longer represented three hours.

In [ ]:
required = ['Date Time', 'T (degC)', 'p (mbar)', 'rh (%)']
df = pd.read_csv(DATASET, usecols=required)
df['Date Time'] = pd.to_datetime(df['Date Time'], dayfirst=True, errors='coerce')
for column in ['T (degC)', 'p (mbar)', 'rh (%)']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df.dropna().sort_values('Date Time').drop_duplicates('Date Time').reset_index(drop=True)
df = df[df['rh (%)'].between(0, 100)]

interval_seconds = df['Date Time'].diff().dt.total_seconds().median()
horizon_rows = max(1, round(3 * 3600 / interval_seconds))
df['Previous_Pressure'] = df['p (mbar)'].shift(horizon_rows)
df['Previous_Time'] = df['Date Time'].shift(horizon_rows)
elapsed_hours = (df['Date Time'] - df['Previous_Time']).dt.total_seconds() / 3600
df['Pressure_Rate'] = (df['p (mbar)'] - df['Previous_Pressure']) / elapsed_hours

for column in ['T (degC)', 'p (mbar)']:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    df = df[df[column].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]

sample = df.iloc[::max(1, len(df) // 100000)].head(100000).copy()
print(f'Prepared {len(df):,} rows; training sample {len(sample):,} rows')
print('Low-humidity prevalence:', f"{(sample['rh (%)'] < 30).mean():.3%}")

In [ ]:
def split_classifier(x, y):
    x_build, x_test, y_build, y_test = train_test_split(x, y, test_size=.20, random_state=RANDOM_STATE, stratify=y)
    x_train, x_val, y_train, y_val = train_test_split(x_build, y_build, test_size=.25, random_state=RANDOM_STATE, stratify=y_build)
    return x_train, x_val, x_test, y_train, y_val, y_test

def best_f1_threshold(model, x_val, y_val):
    probability = model.predict_proba(x_val)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_val, probability)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.nanargmax(f1)])

def classifier_report(model, x_test, y_test, threshold):
    probability = model.predict_proba(x_test)[:, 1]
    predicted = (probability >= threshold).astype(int)
    print(classification_report(y_test, predicted, digits=4, zero_division=0))
    print('ROC-AUC:', round(roc_auc_score(y_test, probability), 4), 'decision threshold:', round(threshold, 6))

## Task 1 — Predict relative humidity from temperature and pressure

In [ ]:
x = sample[['T (degC)', 'p (mbar)']].to_numpy()
y = sample['rh (%)'].to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.30, random_state=RANDOM_STATE)
humidity_model = Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]).fit(x_train, y_train)
prediction = humidity_model.predict(x_test)
print('R2:', round(r2_score(y_test, prediction), 4), 'MAE:', round(mean_absolute_error(y_test, prediction), 4), 'percentage points')
joblib.dump({'task':'humidity_regression','features':['temperature_c','pressure_mbar'],'model_version':'2.0.0','model':humidity_model}, MODEL_DIR / 'humidity_regression_v2.pkl')

## Task 2 — Predict whether relative humidity is below 30%

In [ ]:
y = (sample['rh (%)'] < 30).astype(int).to_numpy()
x_train, x_val, x_test, y_train, y_val, y_test = split_classifier(x, y)
low_humidity_model = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]).fit(x_train, y_train)
low_humidity_threshold = best_f1_threshold(low_humidity_model, x_val, y_val)
classifier_report(low_humidity_model, x_test, y_test, low_humidity_threshold)
joblib.dump({'task':'low_humidity_classifier','features':['temperature_c','pressure_mbar'],'model_version':'2.0.0','decision_threshold':low_humidity_threshold,'model':low_humidity_model}, MODEL_DIR / 'low_humidity_classifier_v2.pkl')

## Task 3 — Predict the pressure-drop fire-risk signal

In [ ]:
pressure_df = df.dropna(subset=['Pressure_Rate']).iloc[::max(1, len(df) // 400000)].head(400000).copy()
pressure_x = pressure_df[['T (degC)', 'Pressure_Rate']].to_numpy()
pressure_y = ((pressure_df['Pressure_Rate'] < -0.5) & (pressure_df['T (degC)'] > 25)).astype(int).to_numpy()
x_train, x_val, x_test, y_train, y_val, y_test = split_classifier(pressure_x, pressure_y)
pressure_model = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]).fit(x_train, y_train)
pressure_threshold = best_f1_threshold(pressure_model, x_val, y_val)
classifier_report(pressure_model, x_test, y_test, pressure_threshold)
joblib.dump({'task':'pressure_risk_classifier','features':['temperature_c','pressure_rate_mbar_per_hour'],'model_version':'2.0.0','decision_threshold':pressure_threshold,'pressure_horizon_hours':3.0,'model':pressure_model}, MODEL_DIR / 'pressure_risk_classifier_v2.pkl')

## One-event inference check
These checks demonstrate the same one-event input used by the dashboard. The Flask service performs this inference without retraining. For Task 3, the service looks up the preceding pressure observation in PostgreSQL.

In [ ]:
event = {'temperature': 32.0, 'pressure_mbar': 995.0}
print('Predicted RH:', round(float(humidity_model.predict([[event['temperature'], event['pressure_mbar']]])[0]), 2), '%')
low_probability = float(low_humidity_model.predict_proba([[event['temperature'], event['pressure_mbar']]])[0, 1])
print('Low-humidity probability:', round(low_probability * 100, 2), '%')
pressure_rate = -1.0  # mbar/hour, normally calculated from the current and prior event
risk_probability = float(pressure_model.predict_proba([[event['temperature'], pressure_rate]])[0, 1])
print('Pressure-drop risk probability:', round(risk_probability * 100, 2), '%')